# **Install necessary libraries**

In [ ]:
# !uv pip install ultralytics
# !uv pip install -U imagecodecs
# import ultralytics
# ultralytics.checks()

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (8 CPUs, 51.0 GB RAM, 43.6/235.7 GB disk)


In [ ]:
# Alterntive solution for numpy related dependency problems
!pip uninstall numpy ultralytics torch torchvision -y
!pip cache purge
!pip install numpy==1.24.3
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install ultralytics
!uv pip install -U imagecodecs
import os
os.kill(os.getpid(), 9)

Found existing installation: numpy 2.4.3
Uninstalling numpy-2.4.3:
  Successfully uninstalled numpy-2.4.3
Found existing installation: ultralytics 8.4.23
Uninstalling ultralytics-8.4.23:
  Successfully uninstalled ultralytics-8.4.23
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Files removed: 0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 84.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to bui

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.5 MB/s eta 0:00:00


Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 71ms
Prepared 1 package in 0.41ms
Uninstalled 1 package in 34ms
Installed 1 package in 19ms
 - numpy==2.3.5
 + numpy==2.4.3


# **Mount the drive for saving the files**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/dataset/code")


In [ ]:
from prediction_to_geodata import yolo_segment_to_shapefile, yolo_obb_to_shapefile

# Train Instance segmentation

In [ ]:
from ultralytics import YOLO
import torch
torch.cuda.is_available()

True

In [ ]:

# Load a model
# model = YOLO('yolo26.yaml')  # build a new model from scratch using model configuraton file
model = YOLO('yolo26n-seg.pt')  # load a pretrained model (recommended for training)
model.to("cuda" if torch.cuda.is_available() else "cpu")  # send a model to a specific device

# Use the model
results = model.train(data="/content/drive/MyDrive/dataset/aoi_112_with_yolo_segment/data.yaml",
                      epochs=30,
                      batch=64,
                      project='/content/drive/MyDrive/dataset/training',
                      name='segment',
                      val=False,
                      resume=False)  # freeze=[1,2,3,4,5, 6, 7, 8, 9]
results = model.val( )  # evaluate model performance on the validation set
# results = model('https://ultralytics.com/images/bus.jpg')  # predict on an image
# results = model.export(format='onnx')  # export the model to ONNX format

engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/dataset/aoi_112_with_yolo_segment/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=segment5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=Tr

# **Perform Prediction on test dataset, convert into geospatial file**

In [ ]:
image_folder = "/content/drive/MyDrive/dataset/aoi_112_with_yolo_mask/images"
output_shapefile = "/content/drive/MyDrive/dataset/aaa_best_yolo_predictions.shp"

model = YOLO('/content/drive/MyDrive/dataset/training/segment4/weights/last.pt')  # load a pretrained YOLO segmentation model
model.to("cuda" if torch.cuda.is_available() else "cpu")
class_names = {0: 'building'}  # Custom mapping

gdf = yolo_segment_to_shapefile(
    model=model,
    image_folder=image_folder,
    ext='tif',
    output_shapefile=output_shapefile,
    conf_threshold=0.1,
    iou_threshold=0.3,
    class_names=class_names
)

# Optional: Display summary statistics
if gdf is not None:
    print("\nSummary Statistics:")
    print(f"Total features: {len(gdf)}")
    print(f"CRS: {gdf.crs}")
    print(f"Columns: {gdf.columns.tolist()}")
    print(f"\nConfidence statistics by class:")
    for class_name in gdf['class_name'].unique():
        class_data = gdf[gdf['class_name'] == class_name]
        print(f"  {class_name}: {len(class_data)} polygons, "
              f"confidence: {class_data['confidence'].mean():.3f} ± {class_data['confidence'].std():.3f}")

Found 215 image files


Processing images: 100%|██████████| 215/215 [00:18<00:00, 11.51it/s]



Results saved to /content/drive/MyDrive/dataset/aaa_best_yolo_predictions.shp
Total polygons: 1552
Classes detected: ['building']
Confidence range: 0.100 - 0.984

Summary Statistics:
Total features: 1552
CRS: PROJCS["WGS 84 / UTM zone 36N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",33],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32636"]]
Columns: ['geometry', 'confidence', 'class_id', 'class_name', 'source_img']

Confidence statistics by class:
  building: 1552 polygons, confidence: 0.402 ± 0.257


# **Train oriented object bounding box detection**

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n-obb.pt')  # load a pretrained model (recommended for training)
model.to("cuda" if torch.cuda.is_available() else "cpu")  # send a model to a specific device
# Use the model
results = model.train(data="/content/drive/MyDrive/dataset/aoi_112_with_yolo_obbox_best/data.yaml",
                      epochs=30,
                      batch=64,
                      project='/content/drive/MyDrive/dataset/training',
                      name='obb',
                      val=False,
                      resume=False)  # freeze=[1,2,3,4,5, 6, 7, 8, 9]
results = model.val( )

engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/dataset/aoi_112_with_yolo_obbox_best/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=obb3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=Tr

# **Perform Oriented bounding box inference then to geodata**

In [ ]:

from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/dataset/training/obb3/weights/best.pt')
model.to("cuda" if torch.cuda.is_available() else "cpu")

image_folder = "/content/drive/MyDrive/dataset/aoi_112_with_yolo_mask/images"
output_shapefile = "/content/drive/MyDrive/dataset/obb_yolo_predictions.shp"
ext = "tif"
class_names = ["building"]

gdf = yolo_obb_to_shapefile(
        model=model,
        image_folder=image_folder,
        ext=ext,
        output_shapefile=output_shapefile,
        conf_threshold=0.25,
        iou_threshold=0.1,
        class_names=class_names
    )

if gdf is not None:
    # Print summary statistics
    print("\n=== Summary Statistics ===")
    print(gdf[['class_name', 'confidence', 'area', 'angle_deg']].describe())

    # Optional: Save to CSV for easy viewing
    csv_path = output_shapefile.replace('.shp', '_summary.csv')
    gdf.drop('geometry', axis=1).to_csv(csv_path, index=False)
    print(f"\nSummary saved to: {csv_path}")

Found 215 image files


Processing images: 100%|██████████| 215/215 [00:09<00:00, 22.31it/s]



Results saved to /content/drive/MyDrive/dataset/obb_yolo_predictions.shp
Total oriented objects: 1541
Classes detected: ['building']
Confidence range: 0.250 - 0.990
Angle range (deg): -41.7 - 47.5

=== Summary Statistics ===
        confidence         area    angle_deg
count  1541.000000  1541.000000  1541.000000
mean      0.734726    32.169444    -6.622910
std       0.229578    30.379521    11.307614
min       0.250078     1.177887   -41.690052
25%       0.541921    15.826679   -13.313050
50%       0.829864    28.455485    -8.662498
75%       0.928218    38.628450    -2.216707
max       0.989985   534.906225    47.462376

Summary saved to: /content/drive/MyDrive/dataset/obb_yolo_predictions_summary.csv
